# 02 · Fundamentals: worked solutions

Each solution includes its own required setup and runs from a fresh kernel. Temporary
files and modules are removed automatically. The checks cover the specified edge
cases, and deliberately invalid examples catch only their expected exception type.


## Exercise 1 · Shared references and shallow copies

`alias` initially points to the same outer list. `shallow` points to a new outer list
whose two elements still refer to the original inner lists. The inner append is
therefore visible through all three names; the outer append is visible only through
`shallow`. Rebinding `alias` later does not change either list object.


In [ ]:
original = [["red"], ["blue"]]
alias = original
shallow = original.copy()
assert alias is original
assert shallow is not original
assert shallow[0] is original[0]
assert shallow[1] is original[1]

shallow[0].append("green")
shallow.append(["black"])
assert original == [["red", "green"], ["blue"]]
assert alias == original
assert shallow == [["red", "green"], ["blue"], ["black"]]
assert len(original) == 2
assert len(shallow) == 3

alias = []
assert alias is not original
assert original == [["red", "green"], ["blue"]]
empty = []
empty_copy = empty.copy()
assert empty_copy == empty and empty_copy is not empty
print(original, shallow, alias)


## Exercise 2 · Duck-typed operations

Numbers interpret `* 0` as multiplication by zero; sequences interpret it as zero
repetitions and yield an empty sequence of the corresponding type. The interface
contract consists of the operations `+` and `*`, not a promise that every pair of
objects can be combined.


In [ ]:
def combine_and_repeat(a, b, count):
    """Add a and b, then multiply/repeat that result by count."""
    return (a + b) * count


assert combine_and_repeat(2, 3, 4) == 20
assert combine_and_repeat("py", "thon", 2) == "pythonpython"
assert combine_and_repeat([1], [2], 0) == []
assert combine_and_repeat(2, 3, 0) == 0
assert combine_and_repeat("a", "b", 0) == ""
assert combine_and_repeat([], [], 3) == []
try:
    combine_and_repeat([1], "2", 1)
except TypeError:
    print("The expected TypeError was raised.")
else:
    raise AssertionError("Incompatible list/string addition should fail.")


## Exercise 3 · Normalize whitespace and case

Splitting without a separator handles repeated spaces, tabs, and newlines. The loop
case-folds each word, and joining uses exactly one space between words. Punctuation
stays attached to its word. Empty input splits to an empty list, whose joined form
is the empty string.


In [ ]:
def normalize_words(text):
    """Collapse whitespace and apply Unicode caseless normalization."""
    normalized = []
    for word in text.split():
        normalized.append(word.casefold())
    return " ".join(normalized)


assert normalize_words("  PYTHON\t Straße \n") == "python strasse"
assert normalize_words("") == ""
assert normalize_words(" \t\n ") == ""
assert normalize_words("Hello, WORLD!") == "hello, world!"
assert normalize_words("a  b   c") == "a b c"
assert normalize_words("Already normalized") == "already normalized"
print(normalize_words("  PYTHON\t Straße \n"))


## Exercise 4 · Report formatting

Each field has an explicit minimum width. The separator itself includes spaces,
which are separate from field padding. Numeric fields align to the right. A long
name expands its field because width is not a length limit.


In [ ]:
def format_report_row(name, count, mean):
    """Format a label, integer count, and numeric mean for a text report."""
    return f"{name:<10} | {count:>4} | {mean:7.2f}"


row = format_report_row("A", 3, 2.5)
assert row == "A          |    3 |    2.50"
assert format_report_row("zero", 0, 0) == "zero       |    0 |    0.00"
assert format_report_row("cold", 2, -1.5) == "cold       |    2 |   -1.50"
assert format_report_row("longer than ten", 10000, 1.25) == (
    "longer than ten | 10000 |    1.25"
)
assert row == "{:<10} | {:>4} | {:7.2f}".format("A", 3, 2.5)
print(row)
print(format_report_row("cold", 2, -1.5))


## Exercise 5 · UTF-8 notes with preserved spaces

Each note receives exactly one newline. `removesuffix("\n")` removes that one line
terminator without stripping surrounding spaces. The exercise contract excludes
embedded newline characters in individual notes. Opening with `w` truncates any
previous content, including when the new note collection is empty.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory


def write_notes(path, notes):
    """Write newline-free strings as UTF-8 lines, replacing existing content."""
    with Path(path).open("w", encoding="utf-8") as stream:
        for note in notes:
            stream.write(note + "\n")


def read_notes(path):
    """Read UTF-8 lines without their terminal newline; preserve other spaces."""
    with Path(path).open("r", encoding="utf-8") as stream:
        return [line.removesuffix("\n") for line in stream]


with TemporaryDirectory() as directory:
    path = Path(directory) / "notes.txt"
    notes = ["Grüße", "  keep spaces  ", ""]
    write_notes(path, notes)
    assert read_notes(path) == notes
    assert path.read_text(encoding="utf-8") == "Grüße\n  keep spaces  \n\n"
    write_notes(path, ["replacement"])
    assert read_notes(path) == ["replacement"]
    write_notes(path, [])
    assert read_notes(path) == []
    assert path.read_text(encoding="utf-8") == ""
    try:
        read_notes(Path(directory) / "missing.txt")
    except FileNotFoundError:
        print("Missing files raise the expected error.")
    else:
        raise AssertionError("Reading a missing path must fail.")
assert not path.exists()
print("Unicode, whitespace, empty notes, replacement, and cleanup checked.")


## Exercise 6 · A script with a quiet import

The main guard protects the demonstration call. Importing still defines the function,
which the importing program can call explicitly. Each subprocess starts fresh,
avoiding import-cache state from a previous cell. Passing arguments as a list avoids
shell parsing; this example does not need a shell.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess
import sys

source = """def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

if __name__ == "__main__":
    print(celsius_to_fahrenheit(0))
"""
with TemporaryDirectory() as directory:
    path = Path(directory) / "temperature_tools.py"
    path.write_text(source, encoding="utf-8")
    script_result = subprocess.run(
        [sys.executable, str(path)],
        capture_output=True, text=True, check=True, timeout=10,
    )
    import_result = subprocess.run(
        [sys.executable, "-c", "import temperature_tools"],
        cwd=directory, capture_output=True, text=True, check=True, timeout=10,
    )
    call_result = subprocess.run(
        [sys.executable, "-c", "import temperature_tools as t; print(t.celsius_to_fahrenheit(-40))"],
        cwd=directory, capture_output=True, text=True, check=True, timeout=10,
    )
    assert script_result.stdout == "32.0\n"
    assert import_result.stdout == ""
    assert call_result.stdout == "-40.0\n"
    assert script_result.stderr == import_result.stderr == call_result.stderr == ""
assert not path.exists()
print("Script:", script_result.stdout.strip())
print("Quiet import checked; explicit call:", call_result.stdout.strip())


## Review answers

Assignment shares an object reference; shallow copying creates a new outer container
with shared references to its contents. `is None` asks about the absence singleton,
whereas `==` asks about values. Duck typing works when the supplied values support
the required operations. `strip` trims ends; `split` partitions text. File reads
advance a cursor, so a second full read at end-of-file is empty. A main guard keeps
a script's demonstration from running during import, and `sys.executable` identifies
the notebook's interpreter.
